In [8]:
!pip install -q mediapipe opencv-python-headless scikit-learn pandas numpy matplotlib gradio ipywidgets

In [9]:
import cv2
import mediapipe as mp
import sklearn
import pandas as pd
import numpy as np
import matplotlib
import gradio as gr
import ipywidgets as widgets

print("All required libraries are installed successfully!")
print("OpenCV:", cv2.__version__)
print("MediaPipe:", mp.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Gradio:", gr.__version__)

All required libraries are installed successfully!
OpenCV: 4.14.0
MediaPipe: 1.1.1
Scikit-learn: 1.6.1
Gradio: 6.30.0


In [ ]:
from google.colab import files

uploaded = files.upload()

video_path = next(iter(uploaded.keys()))

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print("Video:", video_path)
print("FPS:", fps)
print("Total frames:", total_frames)
print("Duration:", total_frames / fps if fps > 0 else "Unknown")

cap.release()
display(Video(video_path, embed=True, width=640))

In [ ]:
import cv2
import mediapipe as mp
from IPython.display import display
from PIL import Image

# Check that your uploaded video is available
print("Video path:", video_path)

cap = cv2.VideoCapture(video_path)
success, frame = cap.read()
cap.release()

if not success:
    raise ValueError("Could not read the video. Check video_path.")

# Open the MediaPipe Face Landmarker
options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.IMAGE,
    num_faces=1
)

with mp.tasks.vision.FaceLandmarker.create_from_options(options) as detector:
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    result = detector.detect(mp_image)

    print("Faces detected:", len(result.face_landmarks))

    if result.face_landmarks:
        print("Landmarks detected:", len(result.face_landmarks[0]))

    display(Image.fromarray(rgb_frame))

In [ ]:

# Facial landmark indices
LEFT_EYE = [33, 160, 158, 133, 153, 144]
RIGHT_EYE = [362, 385, 387, 263, 373, 380]
MOUTH = [13, 14, 78, 308]

def distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

def eye_aspect_ratio(points, indices):
    p = [points[i] for i in indices]
    vertical = distance(p[1], p[5]) + distance(p[2], p[4])
    horizontal = 2 * distance(p[0], p[3])
    return vertical / horizontal if horizontal > 0 else 0

def mouth_aspect_ratio(points):
    vertical = distance(points[13], points[14])
    horizontal = distance(points[78], points[308])
    return vertical / horizontal if horizontal > 0 else 0

# Open video
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
if not fps or fps <= 0:
    fps = 30

options = mp.tasks.vision.FaceLandmarkerOptions(
    base_options=mp.tasks.BaseOptions(
        model_asset_path="face_landmarker.task"
    ),
    running_mode=mp.tasks.vision.RunningMode.VIDEO,
    num_faces=1
)

rows = []
frame_number = 0

with mp.tasks.vision.FaceLandmarker.create_from_options(options) as detector:
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        # Analyze approximately 5 frames per second
        if frame_number % max(1, round(fps / 5)) == 0:
            rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            image = mp.Image(
                image_format=mp.ImageFormat.SRGB,
                data=rgb
            )

            timestamp_ms = int(frame_number * 1000 / fps)
            result = detector.detect_for_video(image, timestamp_ms)

            ear = np.nan
            mar = np.nan
            face_found = len(result.face_landmarks) > 0

            if face_found:
                lm = result.face_landmarks[0]
                points = [(p.x, p.y) for p in lm]

                left_ear = eye_aspect_ratio(points, LEFT_EYE)
                right_ear = eye_aspect_ratio(points, RIGHT_EYE)
                ear = (left_ear + right_ear) / 2
                mar = mouth_aspect_ratio(points)

            rows.append({
                "time_seconds": frame_number / fps,
                "face_detected": face_found,
                "EAR": ear,
                "MAR": mar
            })

        frame_number += 1

cap.release()

df = pd.DataFrame(rows)

if df.empty:
    print("No frames were processed. Check the video file.")
else:
    print("Video analysis completed!")
    print("Frames sampled:", len(df))
    print("Faces detected:", int(df["face_detected"].sum()))
    display(df.head(10))

In [ ]:
import matplotlib.pyplot as plt

if not df.empty:
    plt.figure(figsize=(12, 4))
    plt.plot(df["time_seconds"], df["EAR"], label="Eye Aspect Ratio (EAR)")
    plt.xlabel("Time (seconds)")
    plt.ylabel("EAR")
    plt.title("Eye Openness Indicator Over Time")
    plt.legend()
    plt.grid(True)
    plt.show()

    plt.figure(figsize=(12, 4))
    plt.plot(df["time_seconds"], df["MAR"], label="Mouth Aspect Ratio (MAR)")
    plt.xlabel("Time (seconds)")
    plt.ylabel("MAR")
    plt.title("Mouth Opening Indicator Over Time")
    plt.legend()
    plt.grid(True)
    plt.show()

In [ ]:


# Basic demonstration thresholds — tune them for your video
EAR_THRESHOLD = 0.20
MAR_THRESHOLD = 0.60

df["low_eye"] = df["EAR"] < EAR_THRESHOLD
df["mouth_open"] = df["MAR"] > MAR_THRESHOLD

# Group consecutive sampled frames into events
# Video was sampled at approximately 5 frames per second
SAMPLE_INTERVAL = 0.2

def count_events(series, minimum_seconds=0.6):
    values = series.fillna(False).astype(bool).to_numpy()
    min_samples = max(1, int(minimum_seconds / SAMPLE_INTERVAL))
    count = 0
    run = 0

    for value in values:
        if value:
            run += 1
        else:
            if run >= min_samples:
                count += 1
            run = 0

    if run >= min_samples:
        count += 1

    return count

low_eye_events = count_events(df["low_eye"], 0.6)
mouth_open_events = count_events(df["mouth_open"], 0.6)

print("Possible prolonged eye-closure events:", low_eye_events)
print("Possible mouth-opening events:", mouth_open_events)
print("Note: These are preliminary video indicators, not a diagnosis.")

In [ ]:
# Simulated input: change this value to test the decision logic.
# It is NOT a real alcohol measurement.
simulated_alcohol_level = 0.0  # demonstration units only
SIMULATED_ALCOHOL_THRESHOLD = 0.5

# Use a cautious demo rule based on the fraction of frames with low EAR.
valid_ear = df["EAR"].dropna()

if len(valid_ear) == 0:
    low_eye_fraction = 0.0
else:
    low_eye_fraction = (valid_ear < EAR_THRESHOLD).mean()

possible_drowsiness = low_eye_fraction >= 0.20
alcohol_suspected = (
    simulated_alcohol_level >= SIMULATED_ALCOHOL_THRESHOLD
)

if possible_drowsiness and alcohol_suspected:
    state = "Drowsy + simulated alcohol alert"
elif alcohol_suspected:
    state = "Simulated alcohol alert"
elif possible_drowsiness:
    state = "Possible drowsiness alert"
else:
    state = "No alert triggered by demo rules"

print("Low-EAR frame fraction:", round(low_eye_fraction, 3))
print("Simulated alcohol input:", simulated_alcohol_level)
print("System state:", state)

if state != "No alert triggered by demo rules":
    print("Safety response: display warning and recommend a safe stop.")
else:
    print("Safety response: continue monitoring.")

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

alcohol_input = widgets.FloatSlider(
    value=0.0,
    min=0.0,
    max=1.0,
    step=0.05,
    description="Simulated:",
    continuous_update=False
)

output = widgets.Output()

def update_alcohol(change):
    with output:
        clear_output(wait=True)

        reading = change["new"]
        demo_threshold = 0.5

        print(f"Simulated sensor reading: {reading:.2f}")

        if reading >= demo_threshold:
            print("ALERT: Simulated alcohol threshold exceeded")
            print("Response: Display warning and recommend a safe stop.")
        else:
            print("No simulated alcohol alert triggered.")

        print("\nThis is demonstration data, not a BAC value.")
        print("It does not measure or confirm alcohol consumption.")

alcohol_input.observe(update_alcohol, names="value")
display(alcohol_input, output)
update_alcohol({"new": alcohol_input.value})

In [ ]:
%pip install -q pyttsx3


In [ ]:
%pip install -q gTTS

In [ ]:
from gtts import gTTS
from IPython.display import Audio, display
import io

def voice_warning(message):
    speech = gTTS(text=message, lang="en")
    audio_bytes = io.BytesIO()
    speech.write_to_fp(audio_bytes)
    audio_bytes.seek(0)

    display(Audio(
        data=audio_bytes.read(),
        autoplay=False
    ))

print("Voice warning system is ready!")

In [ ]:
if possible_drowsiness and alcohol_suspected:
    warning = (
        "Warning. Possible driver drowsiness and an alcohol sensor alert. "
        "Please stop driving safely and seek assistance."
    )

elif alcohol_suspected:
    warning = (
        "Warning. The simulated alcohol sensor alert has been triggered. "
        "Do not drive if you may be impaired. Stop safely and seek assistance."
    )

elif possible_drowsiness:
    warning = (
        "Warning. Possible driver drowsiness detected. "
        "Please stop at a safe location and rest."
    )

else:
    warning = "Monitoring active. No warning triggered by the demonstration rules."

print("System message:", warning)
voice_warning(warning)

In [ ]:
%pip install -q gradio

In [ ]:

import gradio as gr
import numpy as np
import pandas as pd
from gtts import gTTS
import tempfile
import os

def run_dashboard(alcohol_level):
    # Check that video analysis results exist
    if "df" not in globals() or df.empty:
        return "No video results found. Run Step 3 first.", None

    ear_values = df["EAR"].dropna()
    if len(ear_values) == 0:
        return "No usable facial landmarks found. Try another video.", None

    # Demonstration rules; tune and validate before research use
    low_eye_fraction = float((ear_values < 0.20).mean())
    possible_drowsiness = low_eye_fraction >= 0.20
    alcohol_suspected = alcohol_level >= 0.5

    if possible_drowsiness and alcohol_suspected:
        state = "COMBINED ALERT"
        message = (
            "Warning. Possible drowsiness and a simulated alcohol alert. "
            "Stop safely and seek assistance."
        )
    elif alcohol_suspected:
        state = "SIMULATED ALCOHOL ALERT"
        message = (
            "Warning. The simulated alcohol threshold was exceeded. "
            "Do not drive if you may be impaired. Stop safely."
        )
    elif possible_drowsiness:
        state = "POSSIBLE DROWSINESS ALERT"
        message = (
            "Warning. Possible drowsiness indicator detected. "
            "Stop at a safe location and rest."
        )
    else:
        state = "NO DEMO ALERT"
        message = (
            "No alert was triggered by the demonstration rules. "
            "This does not establish that driving is safe."
        )

    summary = (
        f"System status: {state}\n"
        f"Low-EAR frame fraction: {low_eye_fraction:.1%}\n"
        f"Simulated alcohol input: {alcohol_level:.2f}\n\n"
        f"Safety message: {message}\n\n"
        "Note: Alcohol input is simulated, not measured. "
        "This prototype does not determine fitness to drive."
    )

    # Generate speech audio for playback in the dashboard
    try:
        tts = gTTS(text=message, lang="en")
        temp = tempfile.NamedTemporaryFile(
            delete=False, suffix=".mp3"
        )
        temp.close()
        tts.save(temp.name)
        audio_path = temp.name
    except Exception as e:
        summary += f"\n\nVoice generation failed: {e}"
        audio_path = None

    return summary, audio_path


with gr.Blocks(title="Driver Safety Monitoring Demo") as demo:
    gr.Markdown("# Driver Impairment Detection System")
    gr.Markdown(
        "Video-based facial indicators + simulated alcohol input + voice warning"
    )

    alcohol_slider = gr.Slider(
        minimum=0,
        maximum=1,
        value=0,
        step=0.05,
        label="Simulated alcohol sensor reading (demonstration only)"
    )

    check_button = gr.Button("Analyze and Generate Warning")
    status_output = gr.Textbox(
        label="Safety System Status",
        lines=8
    )
    voice_output = gr.Audio(
        label="Voice Warning",
        type="filepath",
        autoplay=False
    )

    check_button.click(
        fn=run_dashboard,
        inputs=alcohol_slider,
        outputs=[status_output, voice_output]
    )

demo.launch(share=True)


In [ ]:
from google.colab import files

output_file = "driver_analysis_results.csv"

df.to_csv(output_file, index=False)

print("CSV file created:", output_file)
files.download(output_file)

In [ ]:
summary = {
    "Total sampled frames": len(df),
    "Frames with face detected": int(df["face_detected"].sum()),
    "Face detection rate (%)": round(
        100 * df["face_detected"].mean(), 2
    ),
    "Mean EAR": round(df["EAR"].mean(), 4),
    "Mean MAR": round(df["MAR"].mean(), 4),
    "Possible prolonged eye-closure events": low_eye_events,
    "Possible mouth-opening events": mouth_open_events
}

for key, value in summary.items():
    print(f"{key}: {value}")